# Deliverable 2
2026-06-21

Begin test assessments and complete Group 1 "required" tests.

| Stage | Stage specifications | Time required | Start-End |
| ----- | -------------------- | ------------- | --------- |
| 1 | Learn about the community and note shortcomings or additional toolboxes, such that the wheel need not be reinvented. | 3 weeks | May 1 - May 24 |
| **2** | **Begin test assessments and complete Group 1 “required” tests.** | **4 weeks** | **May 25 - June 21** |
| 3 | Complete Group 2 “Strongly Recommended” tests. | 2 weeks | June 22 - July 5 |
| 4 | Complete Group 3 “Suggested” tests. | 3 weeks | July 6 - July 26 |
| 5 | Explore other tests and finalize documentation.| 3 weeks | July 27 - August 16 |

Therefore, the objectives for this deliverable are to assess each of the following tests in both the QARTOD data manual and in the toolbox itself.

1. Timing/Gap Test
2. Syntax Test
3. Location Test
4. Gross Range Test
    * Including similar Climatological Test (strongly recommended)
    * "Seasonal expectations" variation on the gross range test.
    * Allen et al. 2012 - Variability Generalixed Digital Environmental Model
    * Boyer et al. 2013 - WOD
5. Pressure Test
    * Including similar Density Inversion Test (suggested)
    * Pressure is looking for a monotonically ordered record.
    * This is distinguished by checking potential density $\sigma _{\theta}$ increases with increasing depth. Easily done with `gsw`.
    * Could bundle this with test 8 - Rate of Change.

In [6]:
#   Imports of required packages/modules/libraries for use in this deliverable's tweaks.
#   For handling more generic types (see https://docs.python.org/3/library/stdtypes.html)
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from collections.abc import Sequence    #   list, tuple, range are the three basic sequence types.
    from numbers import Real                #   Real numbers, including `int` and `float` but not complex or imaginary numbers.

import numpy as np
import xarray as xr

In [7]:
#   Import a testing dataset
#   OG1 is OK?
fpath = "/home/aaron-mau/Data/OG1/delayed_SEA056_M102.nc"
ds = xr.load_dataset(fpath)

In [8]:
class QartodFlags:
    """Primary flags for QARTOD."""

    GOOD = 1
    UNKNOWN = 2
    SUSPECT = 3
    FAIL = 4
    MISSING = 9
FLAGS = QartodFlags  # Default name for all check modules
NOTEVAL_VALUE = QartodFlags.UNKNOWN

## Timing/Gap Test
* How is it described in the manual
* How is it represented in the code
* Does anything need to be done in either?

### Manual description of timing/gap test
It's a **required** test, so it's basically one of the most important checks that can be done on the data. Which makes sense - time is one of the dimensions of the data. If you don't know when the data was taken, it loses much of its value.

The manual describes it as "Check for arrival of data".
> Test determines that the most recent profile has been received within the expected time window
> (TIM_INC) and has the correct time stamp (TIM_STMP).
> 
> **Note:** For those gliders that do not update at regular intervals, a large value for TIM_STMP can be
> assigned. The gap check is not a solution for all timing errors. Data could be measured or received
> earlier than expected. This test does not address all clock drift/jump issues.

This is looking at expected time windows `(TIME_INC)` and has an appropriate time stamp `(TIM_STMP)`. In the example, `TIM_INC` is set to 6 hours.

Only 2 flags to assign: Either it passes the test (flag = 1) or it fails by not falling within the specified reporting range (flag = 4).

> `if NOW - TIM_STMP > TIM_INC, flag = 4`

In the manual, it isn't super clear what NOW is supposed to represent. It could be representing real-time applications, but in reality, it's probably just looking at the sequential data points. `NOW - TIM_STMP` is probably meant to be representing sequential data points, rather than whatever `TIM_STMP` value is supposed to be.

In this confusion, I had a discussion with one of my mentors. While it is doing two things, this doesn't apply to instances *not* in a real-time application. The fact that it suggests it is doing two tests in one could create compounding problems:
* The data has a good timestamp and is received in the designated time window (two successes, passes)
* The data has a good timestamp but isn't received in the designated time window (one success, fails)
* The data has a **bad** timestamp and isn't received in the designated time window (one success, fails)
* The data has a **bad** timestamp but **is** received within the time window (no successes, fails)

#### Community understanding of the test
Pinging the UG2 community, I got a couple of responses:
* There are users for the timing/gap test but they question it's usuefulness. Pilots will already know why data is delayed. They would tweak the test during certain moments, like when data transmission was paused, but even then they were still considered "near real-time".
* Another user describes it as essential, but self-implemented. If these checks fail, they don't generate the NetCDF that would go to the DAC.

In other meetings with QARTOD users, they've described this test as a problem due to the interpretation of exactly how it works. There hasn't been a community-wide consensus on how it is meant to be implemented, so that's likely why it's absent in QARTOD.

There *is* a GitHub issue about it (a couple of pull requests, actually), which get at this.

### Code for timing/gap test
I did a quick search for the manual terms, and there's already a disconnect. There is no term for `TIM_STMP` or `TIM_INC` in the entire `ioos_qc` package. I think this is worth bringing up, as users will probably investigate in the same way that I am now. Looking for the terminology that is described in the manual within the code to see how the parameters match up.

Furthermore, there is no defined function or class within `qartod.py` that describes something representing a timing or gap test.
* `utils.py` has a "check_timestamps" function. However, the docstring explicitly says "This is not a QARTOD test, but rather a utility test to make sure **times are in the proper order** and optionally **do not have large gaps prior to processing the data**. It takes in a `max_time_interval` value and returns a boolean. It does not return flags - rather a single boolean of pass/fail.

**Question for Filipe**: We want to reuse code as much as possible, as defined in the GSoC project proposal. Does it make sense to "graduate" this function, or are we sacrificing utility? This `utils.check_timestamps` function is called in `test_utils.py`, so I don't know how it is actually used (if at all).
* It also *adds* a function of confirming that the times are in the proper order. This may not be desireable or in the scope of this description in the manual. **Question for Filipe**: Is that itself worthy of an additional test?
* **A:** It's OK to redefine a new test or series of tests within QARTOD for simplicity. If the existing material in `utils` is not being used extensively, it possibly fills a use case that isn't intended for QARTOD. **A:** If you have additional functionalities you'd like to add, feel free to break them out and test them.

**Question for Filipe**: This is running on time, which is often a coordinate. Do you envision needing to do anything differently here? 
* **A:** No.

**Question for Filipe**: Is there a timestamp format that we should be expecting? Should it be homogenous in the notebook?
* **A:** A good way to get started is using the built-in numpy [datetime64 format](https://numpy.org/doc/stable/reference/arrays.datetime.html).

### Actions for timing/gap test
* (For external workgroup) I think that the manual's description of the test could be improved. "Check for arrival of data" $\rightarrow$ "Check data reporting period".
    * Filipe agrees. The QARTOD working group should go over what this test is intended for in the manual and consider a revision to prevent future confusion.
* Build a new series of tests that fits the description in the manual (without having compounding problems), and an additional test that checks for gaps *within the data itself*. This means:
    1. `impossible_date_test` (similar to GSTPP test 1.2 - timestamps shouldn't be prehistoric) (this should only be flags 1 and 4)
    2. `data_reception_test` (timestamps should be recent for NRT applications) (as per the manual, this is only flags 1 and 4)
    3. `time_gap_test` (designate a window of a certain size and flag points following said gap)

#### impossible_date_test()
Since this is my first change that I'll be proposing to the QARTOD QC, we'll want to preserve the existing codebase and make sure that everything is consistent. This means the docstrings at the beginning of the function should be similar to other existing tests, typing should be consistent, and the flag exports should be the same general item.

There are a few notes to make prior to starting this test, however.

##### Numpy masked arrays
Most of the other tests use numpy **masked** arrays, of which I wasn't particularly familiar with.

[Masked arrays](https://numpy.org/doc/stable/reference/maskedarray.generic.html) seem to be handy because they consist of the data and a mask. Kind of like a way to track a condition on each point in a data array.

In the documentation, the following example is given:
```python
import numpy as np
import numpy.ma as ma
y = ma.array([1, 2, 3], mask = [0, 1, 0])
```

In this case, the second point (index 1) is 1, meaning that this element is invalid. It's a great way to keep track of tests that are run on data (pass/fail, yes/no).

These arrays are commonly referenced using `y.data` or `y.mask` for the values and mask, respectively. Cool! I haven't worked very much with this before and it's much simpler than assigning multiple ndarrays to handle the same thing.

##### Regarding datetime64
[Numpy's datetime64](https://numpy.org/doc/stable/reference/arrays.datetime.html) is handy for a lot of things, but keep in mind that it is usually formatted as a timestamp *since Jan 1st, 1970*. The year, month, day, and minute can all be pulled out with some simple string formatting (like `"datetime64[Y]"), but you'll need to convert these items relative to 1970. *Note that you should convert it to an integer before using it.*
* datetime64[Y] $\rightarrow$ add '1970' to it.
* datetime64[M] $\rightarrow$ mod 12 and add 1 for the calendar month.
* datetime64[h] $\rightarrow$ mod 24 gives you the hour of the day.
* datetime64[m] $\rightarrow$ mod 60 gives you the minute of the hour.

For the dates in the month, you've got cases like leap years which we'll want to account for which adds a 29th day to February. We can get how long each month is using each calendar month, converting that to the day, and doing the same for the next month's start. Then the difference between them should be how long the month is, and which valid days there are.

`first day of March - first day of February = length of February`

Furthermore, we want the day of the month to be valid. We could approach this from a complicated series of if statements based on Julian date and the current year, calculating out each month independently, but we can also convert from months to days using Numpy's `astype` arguments.

In [9]:
def impossible_date_test(
        tinp: Sequence[Real],
        fail_span: tuple[Real, Real] | None = None,
) -> np.ma.core.MaskedArray:
    """
    This test confirms that the date and time for the data are reasonable.
    
    Given an array of time data, this test breaks the data down into a series of sub-tests.
    These tests are similar to those outlined in test 1.2 of the GTSPP RTQC Manual (IOC, 2010).
    * The year is either present or in the past.
    * The month is within 1-12.
    * The day is possible for the given month, depending on year.
    * The hour of the day is between 0-23.
    * The minutes are between 0-50.
    * (Optional) The time data is within a user-defined tolerance.
    Data deemed to have failed any or all of these tests are flagged as FAIL. Any missing and masked data is flagged as UNKNOWN.

    Parameters
    ----------
    tinp
        Time input data as a numeric numpy array or list of real numbers.
    fail_span
        2-tuple range which to flag outside data as FAIL. [optional]

    Returns
    -------
    flag_arr
        A masked array of flag values equal in size to that of the input `tinp`.
    """
    #   Init 
    original_shape = tinp.shape
    tinp = np.ma.asarray(tinp, dtype="datetime64[ns]").flatten()
    flag_arr = np.ma.ones(tinp.size, dtype="uint8") #   Init to flag 1 "good"
    
    tinp.mask = np.isnat(tinp.data)
    flag_arr[tinp.mask] = QartodFlags.MISSING   #   Init missing timestamps to the missing flag
    valid = ~tinp.mask  #   Define where the data point are not missing, such that we can index them and keep those flags.

    #   Year test - where year is in the future
    yr = tinp.data.astype("datetime64[Y]").astype(int) + 1970
    current_yr = np.datetime64("now", "Y").astype(int) + 1970
    flag_arr[valid & (yr > current_yr)] = QartodFlags.FAIL

    # #   Month test - the month must be 1-12
    # mn = tinp.data.astype("datetime64[M]").astype(int) % 12 + 1
    # flag_arr[valid & ((mn < 1) | (mn > 12))] = QartodFlags.FAIL
    
    # #   Day test - day must be possible for the given month
    # #   Define the start of the first and next month for all points in the array
    # mn_1 = tinp.data.astype("datetime64[M]").astype("datetime64[D]")
    # mn_2 = (tinp.data.astype("datetime64[M]") + 1).astype("datetime64[D]")
    # days = (mn_2 - mn_1).astype(int)
    # #   Day of the month
    # dy   = (tinp.data.astype("datetime64[D]") - mn_1).astype(int) + 1
    # flag_arr[valid & ((dy < 1) | (dy > days))] = QartodFlags.FAIL

    # #   Hour test - must be 0-23
    # hr = tinp.data.astype("datetime64[h]").astype(int) % 24
    # flag_arr[valid & ((hr < 0) | (hr > 23))] = QartodFlags.FAIL

    # #   Minute test - must be 0-59
    # mi = tinp.data.astype("datetime64[m]").astype(int) % 60
    # flag_arr[valid & ((mi < 0) | (mi > 59))] = QartodFlags.FAIL

    #   Span test
    if fail_span is not None:
        #   Convert if not already
        low, high = np.datetime64(fail_span[0]), np.datetime64(fail_span[1])
        flag_arr[valid & ((tinp.data < low) | (tinp.data > high))] = QartodFlags.FAIL
    
    return flag_arr.reshape(original_shape)

Discovery while testing: Numpy datetime64 doesn't actually allow months, days, hours, or minutes to be outside of tolerated ranges.

Inserting a fake datetime throws a ValueError:
```
np.datetime64("2025-12-32")
ValueError: Day out of range in datetime string "2025-12-32"

np.datetime64("2026-01-12T17:61:07.000000000")
ValueError: Minutes out of range in datetime string "2026-01-12T17:61:07.000000000"
```
As such, we can comment out many of the checks we defined. It even handles leap days.

```
np.datetime64("2024-02-29") #   This was a leap day

np.datetime64("2025-02-29") #   Most certainly not a leap day
ValueError: Day out of range in datetime string "2025-02-29"
```
The minutia described in the GTSPP is handled by datetime64 - if there are impossible values with the date and time then it will error out before getting tested and that tends to be easy to find.

Therefore, the most important aspect of this is probably the span test and checking for the future year. This can be further simplified by just looking for dates in the future, so we won't need to break out the year at all.

The final function to be added should then be

In [10]:
def impossible_date_test(
        tinp: Sequence[Real],
        fail_span: tuple[Real, Real] | None = None,
) -> np.ma.core.MaskedArray:
    """
    This test confirms that the date and time for the data are reasonable.
    
    Given an array of time data, this test breaks the data down into a series of sub-tests.
    These tests are similar to those outlined in test 1.2 of the GTSPP RTQC Manual (IOC, 2010).
    * The year is either present or in the past.
    * (Optional) The time data is within a user-defined tolerance.
    Data deemed to have failed any or all of these tests are flagged as FAIL. Any missing and masked data is flagged as UNKNOWN.

    Parameters
    ----------
    tinp
        Time input data as a numeric numpy array or list of real numbers.
    fail_span
        2-tuple range which to flag outside data as FAIL. [optional]

    Returns
    -------
    flag_arr
        A masked array of flag values equal in size to that of the input `tinp`.
    """
    #   Init 
    original_shape = tinp.shape
    tinp = np.ma.asarray(tinp, dtype="datetime64[ns]").flatten()
    flag_arr = np.ma.ones(tinp.size, dtype="uint8") #   Init to flag 1 "good"
    
    tinp.mask = np.isnat(tinp.data)
    flag_arr[tinp.mask] = QartodFlags.MISSING   #   Init missing timestamps to the missing flag
    valid = ~tinp.mask  #   Define where the data point are not missing, such that we can index them and keep those flags.

    #   Check for time travelers
    now = np.datetime64("now")
    flag_arr[valid & (tinp.data > now)] = QartodFlags.FAIL

    #   Span test
    if fail_span is not None:
        #   Convert if not already
        low, high = np.datetime64(fail_span[0]), np.datetime64(fail_span[1])
        flag_arr[valid & ((tinp.data < low) | (tinp.data > high))] = QartodFlags.FAIL
    
    return flag_arr.reshape(original_shape)

Let's confirm that the checker works. Primarily, this is confirming that there are no NaT values, future dates, or anything outside of our span.

In [11]:
flags = impossible_date_test(ds.TIME.to_numpy(), fail_span = ("2020-01-01T00:00:00.0000", "2027-01-01T00:00:00.0000"))
print(any(flags == QartodFlags.FAIL))
print(any(flags == QartodFlags.MISSING))

False
False


Let's insert a bad data point in the future and confirm that we get one bad point out.

In [12]:
bad_data = ds.TIME.to_numpy().copy()
bad_data[20000] = np.datetime64("2088-01-12T23:05:16.000000000")    #   (Cheers to the QARTOD users in 2088)
print(bad_data[19998:20002])

['2026-01-12T23:05:14.000000000' '2026-01-12T23:05:15.000000000'
 '2088-01-12T23:05:16.000000000' '2026-01-12T23:05:17.000000000']


In [13]:
flags = impossible_date_test(bad_data)
print(any(flags == QartodFlags.FAIL))
print(np.where(flags == QartodFlags.FAIL)[0][0])    #   Return the index - np.where nests things in a tuple of arrays

True
20000


And we'll modify the data further by inserting a chunk of dates that are outside of our `fail_span`. We'll insert a lot, so let's just return the first and last 5 indexes where the flags are bad. We should still see the bad time from before at index 20000.

In [14]:
bad_data[1500:2200] = np.datetime64("2013-01-12T23:05:16.000000000")
flags = impossible_date_test(bad_data, fail_span=("2020-01-01T00:00:00.0000","2027-01-01T00:00:00.0000"))
idx = np.where(flags == QartodFlags.FAIL)[0]
print(idx[:5],idx[-5:])

[1500 1501 1502 1503 1504] [ 2196  2197  2198  2199 20000]


#### data_reception_test()
This is a pretty straightforward test that should be more useful to datacenters that are expecting data within a certain timeframe. In many ways, it seems like a subset of the previous test. In the manual, the time window is described as `TIM_INC`, where the example sets it to 6 hours. In this case, our `fail_span` can be a single value, in hours, which if the data is older than by that amount, will cause it to be flagged.

It might be helpful to define a `starting timestamp` value of some sort (I'll call it `from_time`), such that we can check if data were received within X hours of timestamp Y. This isn't explicitly stated in the manual, so let's have it be optional and default to `np.datetime64("now")`.

In [15]:
def data_reception_test(
        tinp: Sequence[Real],
        fail_span: Real = 6,
        from_time: Real | None = None,
) -> np.ma.core.MaskedArray:
    """
    This test checks for data timestamps to be within a certain amount of time of present.
    
    Timestamps that are further away in time from the `from_time` variable than that designated
    by `fail_span` are flagged as FAIL. Data points that are newer than `from_time` are not considered
    for this test and are passed through as GOOD. Any missing and masked data is flagged as UNKNOWN.

    Parameters
    ----------
    tinp
        Time input data as a numeric numpy array or list of real numbers.
    fail_span
        A numeric value in hours which, if data is older than, is flagged as FAIL.
    from_time
        A timestamp which, if defined, is used to anchor measurements against. Defaults to current date and time. [optional]

    Returns
    -------
    flag_arr
        A masked array of flag values equal in size to that of the input `tinp`.
    """
    #   Init 
    if from_time == None:
        from_time = np.datetime64("now")
    else:
        from_time = np.datetime64(from_time)
    
    original_shape = tinp.shape
    tinp = np.ma.asarray(tinp, dtype="datetime64[ns]").flatten()
    flag_arr = np.ma.ones(tinp.size, dtype="uint8") #   Init to flag 1 "good"
    
    tinp.mask = np.isnat(tinp.data)
    flag_arr[tinp.mask] = QartodFlags.MISSING   #   Init missing timestamps to the missing flag
    valid = ~tinp.mask  #   Define where the data point are not missing, such that we can index them and keep those flags.
    
    fail_span = np.timedelta64(int(fail_span * 3600), "s")
    diff_time = from_time - tinp.data
    flag_arr[valid & (diff_time > fail_span)] = QartodFlags.FAIL
    
    return flag_arr.reshape(original_shape) 

Let's test it with an arbitrary value inside the dataset, using the default 6 hour tolerance for flagging. Everything after this 6 hour tolerance should be flagged as 1, whereas everything before it should be flagged as 4.

In [16]:
from_time = np.datetime64('2026-02-10T11:17:07.000000000')
print(np.where(ds.TIME == from_time))

(array([1690309]),)


In [17]:
flags = data_reception_test(ds.TIME.to_numpy(), from_time=from_time)
print(f"The last index where data are flagged: {np.where(flags == 4)[0][-1]}")
print(f"Length of the array: {len(ds.TIME)}")

The last index where data are flagged: 1669201
Length of the array: 1690311


In [18]:
print(flags[1669195:1669210])   #   Looks to be index 1669201 is just over 6 hours away.
print(np.datetime64(from_time - ds.TIME[1669202].item()))   #   Returns just 6 hours from Jan 1, 1970

[4 4 4 4 4 4 4 1 1 1 1 1 1 1 1]
1970-01-01T06:00:00.000000000


#### time_gap_test()
This is the first test that probably *shouldn't* be used on NRT data, as it looks at the gaps within the data itself. Similar to the previous test, given a threshold, it takes the diff of the time array and flags the point *after* any gaps that are exceeding the specified `fail_span`.

Since we'll take the difference of the values against each other, the final array will be of size N-1, presenting a [fencepost](https://en.wikipedia.org/wiki/Off-by-one_error#Fencepost_error) problem. This untested point can be left as UNTESTED or UNKNOWN, unless we work backwards and confirm that it is less than `fail_span` from the next point.

In [19]:
def time_gap_test(
    tinp: Sequence[Real],
    fail_span: Real = 2,
) -> np.ma.core.MaskedArray:
    """
    This test checks for gaps in the data time that exceed a specific threshold.
    
    The data `tinp` is differentiated and those changes in time are compared against the time threshold
    `fail_span`. If the difference in time between points exceeds the value of `fail_span`, the following
    data point is flagged as FAIL. The first point inherits the flag of the second point, unless the
    second point is UNKNOWN or MISSING. Any missing and masked data is flagged as UNKNOWN.

    Parameters
    ----------
    tinp
        Time input data as a numeric numpy array or list of real numbers.
    fail_span
        A numeric value for time duration in hours which, if timestamps jump by more than between points, flags following data point as FAIL.

    Returns
    -------
    flag_arr
        A masked array of flag values equal in size to that of the input `tinp`.
    """
    
    original_shape = tinp.shape
    tinp = np.ma.asarray(tinp, dtype="datetime64[ns]").flatten()
    flag_arr = np.ma.ones(tinp.size, dtype="uint8") #   Init to flag 1 "good"
    
    tinp.mask = np.isnat(tinp.data)
    flag_arr[tinp.mask] = QartodFlags.MISSING   #   Init missing timestamps to the missing flag
    valid = ~tinp.mask

    diff_time = np.ma.zeros(tinp.size, dtype="timedelta64[ns]") #   Guarantee that the array is the same size
    diff_time[1:] = tinp.data[1:] - tinp.data[:-1]

    #   NaT testing - guarantee that the previous entry isn't NaT
    valid[1:] &= valid[:-1]

    fail_span = np.timedelta64(int(fail_span * 3600), "s")
    flag_arr[valid & (diff_time > fail_span)] = QartodFlags.FAIL

    #   Handle the first point
    if tinp.mask[1]:
        flag_arr[0] = QartodFlags.UNKNOWN
    else:
        flag_arr[0] = flag_arr[1]

    return flag_arr.reshape(original_shape)

In [20]:
flags = time_gap_test(ds.TIME)
print(any(flags == 4))
print(np.where(flags == 4))

True
(array([1414125, 1449545, 1449929, 1476935, 1512225, 1630895]),)


This is the first test I've run so far where there are actual results on the data I've been testing. In this case, I've got 6 potential gaps in the data (ending at the returned indices above). Let's investigate them a little further.

In [21]:
print(np.diff(ds.TIME[1414122:1414130]))    #   Difference in nanoseconds [ns] ix = 2
print(ds.TIME[1414122:1414130])

[    1000000000     1000000000 76010000000000     1000000000
     1000000000     1000000000     1000000000]
<xarray.DataArray 'TIME' (N_MEASUREMENTS: 8)> Size: 64B
array(['2026-01-29T11:25:48.000000000', '2026-01-29T11:25:49.000000000',
       '2026-01-29T11:25:50.000000000', '2026-01-30T08:32:40.000000000',
       '2026-01-30T08:32:41.000000000', '2026-01-30T08:32:42.000000000',
       '2026-01-30T08:32:43.000000000', '2026-01-30T08:32:44.000000000'],
      dtype='datetime64[ns]')
Coordinates:
  * N_MEASUREMENTS  (N_MEASUREMENTS) int64 64B 1414122 1414123 ... 1414129
    LATITUDE        (N_MEASUREMENTS) float64 64B 55.45 55.45 ... 55.43 55.43
    LONGITUDE       (N_MEASUREMENTS) float64 64B 16.23 16.23 ... 16.28 16.28
    TIME            (N_MEASUREMENTS) datetime64[ns] 64B 2026-01-29T11:25:48 ....
    DEPTH           (N_MEASUREMENTS) float64 64B 1.365 1.293 ... 1.271 1.264
Attributes:
    interpolation_methodology:  
    long_name:                  Time elapsed since 1970-01-01T00:00:

In [22]:
gap = np.diff(ds.TIME[1414122:1414130])[2]
print(gap.astype("timedelta64[h]"))

21 hours


Sure enough, this data has a gap in it of about 21 hours in length. And given that we found it using the flag, we should be good to ID other points with the same issue using the flags.

In [23]:
idx = np.where(flags == 4)[0]
gaps = (ds.TIME.values[idx] - ds.TIME.values[idx - 1]).astype("timedelta64[h]")
print(gaps)
print(ds.TIME.values[idx])

[ 21  20   4 128  13  21]
['2026-01-30T08:32:40.000000000' '2026-01-31T15:30:16.000000000'
 '2026-01-31T20:07:25.000000000' '2026-02-06T12:28:44.000000000'
 '2026-02-07T11:37:37.000000000' '2026-02-09T18:23:44.000000000']


So we can see that there are data gaps of variable length present within the data, including a very long one of 128 hours preceeding February 6th.

There is one last thing to test - to confirm that the first data point is flagged correctly.

In [24]:
bad_data = ds.TIME.copy()
bad_data[1] = np.nan

In [25]:
flags = time_gap_test(bad_data[0:10])
print(flags)

[2 9 1 1 1 1 1 1 1 1]


#### Unit testing
Now! To take these tests that we've done on each of the timing/gap tests and get them incorporated into the **unit** tests.

Unit tests are helpful because they:
* Confirm that the code does what we expect it to do in a controlled environment.
* To a certain extent, documentation. If your tests are well thought-out, users will almost have examples of the code in the tests.
* Demonstrate robustness in the code, should Python functions update or if you [refactor anything](https://en.wikipedia.org/wiki/Code_refactoring).
* Finding bugs.

There are loads of quotes out there about how good unit tests make for good, reliable code. When adding functionality, it's very important to add or modify unit tests to confirm that the code still works the way you intend it to.

All of these unit tests are typically done in Python using `pytest` or `unittest`. In this instance, I test using `pytest` with a tweaked output.

Navigating to the `ioos_qc` **/tests/** folder, `pytest` is pretty straightforward to call.

```shell
pytest test_qartod.py

============================================================ test session starts =============================================================
platform linux -- Python 3.14.4, pytest-9.0.3, pluggy-1.6.0 -- /home/aaron-mau/miniforge3/envs/gsoc/bin/python3.14
cachedir: .pytest_cache
rootdir: /home/aaron-mau/Code/gsoc/mau/ioos_qc
configfile: pyproject.toml
plugins: anyio-4.13.0, cov-7.1.0
collected 69 items                                                                                                                           

test_qartod.py::QartodLocationTest::test_location PASSED
test_qartod.py::QartodLocationTest::test_location_bad_input PASSED
test_qartod.py::QartodLocationTest::test_location_bbox PASSED
test_qartod.py::QartodLocationTest::test_location_distance_threshold PASSED
test_qartod.py::QartodLocationTest::test_single_location_nan PASSED
test_qartod.py::QartodLocationTest::test_single_location_none PASSED
test_qartod.py::QartodGrossRangeTest::test_gross_range_bad_input PASSED
test_qartod.py::QartodGrossRangeTest::test_gross_range_check PASSED
test_qartod.py::QartodGrossRangeTest::test_gross_range_check_masked PASSED
test_qartod.py::QartodClimatologyPeriodTest::test_climatology_test_periods_day_of_year PASSED
test_qartod.py::QartodClimatologyPeriodTest::test_climatology_test_periods_monthly PASSED
test_qartod.py::QartodClimatologyPeriodTest::test_climatology_test_periods_quarter PASSED
test_qartod.py::QartodClimatologyPeriodTest::test_climatology_test_periods_week_of_year PASSED
test_qartod.py::QartodClimatologyPeriodFullCoverageTest::test_dayofyear_periods PASSED
test_qartod.py::QartodClimatologyPeriodFullCoverageTest::test_monthly_periods PASSED
test_qartod.py::QartodClimatologyPeriodFullCoverageTest::test_quarterly_periods PASSED
test_qartod.py::QartodClimatologyPeriodFullCoverageTest::test_weekofyear_periods PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_fspan_maximum PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_fspan_minimum PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_fspan_out_of_range_high PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_fspan_out_of_range_low PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_tspan_maximum PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_tspan_minimum PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_tspan_out_of_range_high PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_tspan_out_of_range_low PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_vspan_maximum PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_vspan_minimum PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_vspan_out_of_range_high PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_vspan_out_of_range_low PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_zspan_maximum PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_zspan_minimum PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_zspan_out_of_range_high PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_zspan_out_of_range_low PASSED
test_qartod.py::QartodClimatologyDepthTest::test_climatology_test_all_unknown PASSED
test_qartod.py::QartodClimatologyMissingTest::test_climatology_missing_values PASSED
test_qartod.py::QartodClimatologyTest::test_climatology_test PASSED
test_qartod.py::QartodClimatologyTest::test_climatology_test_depths PASSED
test_qartod.py::QartodClimatologyTest::test_climatology_test_fail PASSED
test_qartod.py::QartodClimatologyTest::test_climatology_test_seconds_since_epoch PASSED
test_qartod.py::QartodSpikeTest::test_spike PASSED
test_qartod.py::QartodSpikeTest::test_spike_initial_final_values PASSED
test_qartod.py::QartodSpikeTest::test_spike_masked PASSED
test_qartod.py::QartodSpikeTest::test_spike_methods PASSED
test_qartod.py::QartodSpikeTest::test_spike_negative_vals PASSED
test_qartod.py::QartodSpikeTest::test_spike_realdata PASSED
test_qartod.py::QartodSpikeTest::test_spike_test_bad_method PASSED
test_qartod.py::QartodSpikeTest::test_spike_test_inputs PASSED
test_qartod.py::QartodRateOfChangeTest::test_rate_of_change PASSED
test_qartod.py::QartodRateOfChangeTest::test_rate_of_change_fail_flag PASSED
test_qartod.py::QartodRateOfChangeTest::test_rate_of_change_missing_values PASSED
test_qartod.py::QartodRateOfChangeTest::test_rate_of_change_negative_values PASSED
test_qartod.py::QartodFlatLineTest::test_flat_line PASSED
test_qartod.py::QartodFlatLineTest::test_flat_line_missing_values PASSED
test_qartod.py::QartodFlatLineTest::test_flat_line_short_timeseries PASSED
test_qartod.py::QartodFlatLineTest::test_flat_line_starting_from_beginning PASSED
test_qartod.py::QartodFlatLineTest::test_flat_line_with_spike PASSED
test_qartod.py::QartodAttenuatedSignalTest::test_attenuated_signal PASSED
test_qartod.py::QartodAttenuatedSignalTest::test_attenuated_signal_missing PASSED
test_qartod.py::QartodAttenuatedSignalTest::test_attenuated_signal_missing_time_window PASSED
test_qartod.py::QartodAttenuatedSignalTest::test_attenuated_signal_range PASSED
test_qartod.py::QartodAttenuatedSignalTest::test_attenuated_signal_time_window PASSED
test_qartod.py::QartodDensityInversionTest::test_density_inversion_bad_depth_value PASSED
test_qartod.py::QartodDensityInversionTest::test_density_inversion_down_up_cast_flags PASSED
test_qartod.py::QartodDensityInversionTest::test_density_inversion_downcast_flags PASSED
test_qartod.py::QartodDensityInversionTest::test_density_inversion_input PASSED
test_qartod.py::QartodDensityInversionTest::test_density_inversion_one_record_input PASSED
test_qartod.py::QartodDensityInversionTest::test_density_inversion_stable_depth_flags PASSED
test_qartod.py::QartodDensityInversionTest::test_density_inversion_upcast_flags PASSED
test_qartod.py::QartodUtilsTests::test_qartod_compare PASSED

============================================================= 69 passed in 1.10s =============================================================
```

This lists each of the tests that are accounted for in the `test_qartod.py` code. In this case, there are 69 tests, consisting of various **assertions** (basically a confirmation that the code has done what we're expecting).

This is nice, but we can add to this output by using `cov`, a [method of tracking](https://about.codecov.io/) which lines of code have been tested. For example, each of the tests listed above can be passed, but it's possible that not 100% of the lines in the code are being tested. This is great for finding those edge scenarios, like with lots of `if` statements or data conditions.

I modify the line for `pytest` with additional `cov` arguments:
```shell
pytest test_qartod.py --cov-report=term-missing --cov=ioos_qc

... (the same pytest output as above) ...

______________________________________________ coverage: platform linux, python 3.14.4-final-0 _______________________________________________

Name                                                                             Stmts   Miss  Cover   Missing
--------------------------------------------------------------------------------------------------------------
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/__init__.py                            4      2    50%   5-6
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/_version.py                            1      0   100%
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/argo.py                               51     51     0%   3-154
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/axds.py                               43     43     0%   3-120
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/config.py                            235    235     0%   12-549
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/config_creator/__init__.py             2      2     0%   1-9
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/config_creator/config_creator.py     219    219     0%   1-633
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/config_creator/fx_parser.py           66     66     0%   25-165
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/config_creator/get_assets.py         140    140     0%   3-288
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/config_creator/make_config.py         11     11     0%   1-34
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/gliders.py                             5      5     0%   3-14
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/plotting.py                           60     60     0%   1-182
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/qartod.py                            346     44    87%   61-62, 83-84, 143-144, 153-154, 219-220, 238-239, 302, 311-328, 339-340, 346-347, 352-353, 358-359, 366-368, 456-459, 875-876, 896, 960-961, 968
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/results.py                            87     87     0%   1-181
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/stores.py                            180    180     0%   1-427
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/streams.py                           247    247     0%   1-592
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/utils.py                             141     83    41%   40-41, 46-100, 111-148, 178, 181, 194-196, 220-226, 236-245, 251-253, 258-265, 273-282
--------------------------------------------------------------------------------------------------------------
TOTAL                                                                             1838   1475    20%
```

This covers all of the individual tests in the folder (even though I specified `test_qartod.py`, not sure why that went awry). As you can see, `qartod.py` has about 87% coverage before adding these new methods (and the average for the whole `ioos_qc` toolbox is 20%, but that's not our priority). The specific lines that are untested are given in the "Missing" column. When we add additional methods, we can expect this coverage to drop.

After adding these new methods to the end, we can see that, indeed, coverage drops becuase there are 3 fat regions of code that haven't been tested after line 1024.

```shell
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/qartod.py                            389     84    78%   61-62, 83-84, 143-144, 153-154, 219-220, 238-239, 302, 311-328, 339-340, 346-347, 352-353, 358-359, 366-368, 456-459, 875-876, 896, 960-961, 968, 1024-1042, 1072-1089, 1117-1140
```

In exploring the existing unit tests, it looks like we're using a Class system. As an example:

```python
class QartodSpikeTest(unittest.TestCase):
    def setUp(self):
        self.suspect_threshold = 25
        self.fail_threshold = 50

    def test_spike(self):
        """Test to make ensure single value spike detection works properly."""
        arr = [10, 12, 999.99, 13, 15, 40, 9, 9]

        # First and last elements should always be good data, unless someone
        # has set a threshold to zero.
        expected = [2, 4, 4, 4, 1, 3, 1, 2]

        inputs = [
            arr,
            np.asarray(arr, dtype=np.float64),
            dask_arr(np.asarray(arr, dtype=np.float64)),
        ]
        for i in inputs:
            npt.assert_array_equal(
                qartod.spike_test(
                    inp=i,
                    suspect_threshold=self.suspect_threshold,
                    fail_threshold=self.fail_threshold,
                ),
                expected,
            )
#   much more below
```
This is different from how I've worked on unit tests before, but I can see why this would be helpful.
* Some of the `qartod` methods, like the climatology test, use Classes already. In some ways, this may help mimic their functionality.
* It means you don't need to use *decorators* or [*fixtures*](https://docs.pytest.org/en/stable/how-to/fixtures.html), which allow you to pass fake data around. Here, you can define your data when you initialize the testing class, allowing you to then pass that along to all the follow up tests.
* The alternative, which I've done, is just have lots and lots of individually-named functions. Like the `test_spike` example above, but for each and every scenario I could imagine to get 100% coverage of the original code. Having them all live within a class helps keep things organized.

So let's get started! First thing's first, I've used some data in testing that I could probably reduce in size and copy into my class when it's initialized. This would be `ds.TIME` or `bad_data`, good for testing the different scenarios on each of the new methods we've added. All tests within each class will need to have the "`test_`" prefix.
* impossible_date_test()
    * For this one, we're very reliant on `np.datetime64` handling errors with impossible dates or times. In testing, we should get it to error out. This is what `unittest.TestCase` does for us elsewhere in the code.
* data_reception_test()
* time_gap_test()

In [26]:
import unittest
from ioos_qc import qartod  #   Since I've added the functions to my local version of qartod, my new functions should show up

In [27]:
"impossible_date_test" in dir(qartod)   #   dir(object) lists the things you can do with it. A good way to confirm we're ready to test!

True

In typing up these tests, it becomes clearer that `shape` is a limiting factor on some of my data. At first, I tried
```python
dt = "2026-13-91T00:00:00.000"
```
but neither that nor `list` have the `shape` attribute, so it broke in testing. As such, I have to move forward with a numpy array.
```python
dt = np.array("2026-13-91T00:00:00.000")
```

##### QartodImpossibleDateTest

In [28]:
class QartodImpossibleDateTest(unittest.TestCase):
    def setUp(self):
        """Define the data that we're going to pass into the following tests.
        
        Within the class, these data will live as attributes of 'self'."""
        #   As we defined them before.
        times = ['2026-01-12T23:05:14.000000000','2026-01-12T23:05:15.000000000','2026-01-12T23:05:16.000000000','2026-01-12T23:05:17.000000000']
        self.data_good = np.array(times, dtype="datetime64")
        self.data_bad = self.data_good.copy()
        self.data_bad[1] = np.datetime64("2088-01-12T23:05:16.000000000")
    def test_error_bad_datetime(self):
        dt = np.array("2026-13-91T00:00:00.000")    #   the function will attempt to convet this to datetime64
        self.assertRaises(
            ValueError,
            qartod.impossible_date_test,
            tinp=dt,
        )   #   This is a type of assertion that checks for errors
        dt = np.array("2026-12-00T38:00:00.000")
        self.assertRaises(
            ValueError,
            qartod.impossible_date_test,
            tinp = dt,
        )
    def test_all_nat(self):
        dt = np.full(4, np.datetime64("NaT"))
        flags = qartod.impossible_date_test(tinp = dt)
        assert all(flags == 9)
    def test_future_year(self):
        """Uses data_bad, which has an entry set well in the future."""
        flags = qartod.impossible_date_test(tinp = self.data_bad)
        assert 4 in flags   #   This is a more typical assertion. Here, we expect there to be a bad entry in the flags.
        assert flags[1] == 4
        assert type(flags) == np.ma.core.MaskedArray
    def test_span_good(self):
        flags = qartod.impossible_date_test(tinp = self.data_good, fail_span=("2012-01-01T00:00:00.000", "2027-01-01T00:00:00.000"))
        assert all(flags == 1)


So now that we've made our assertions, we should be able to copy this over to `qartod.py` and try running our unit testing on it.

If I recall correctly, this can be done inside a notebook. Let's confirm that this looks right.

In [29]:
import pytest
testing_path = "/home/aaron-mau/Code/gsoc/mau/ioos_qc/tests/test_qartod.py"
# pytest.main([full])   #   Learning how to run pytest in a jupyter notebook...
pytest.main(["--assert=plain", "-v", testing_path])

============================= test session starts ==============================
platform linux -- Python 3.14.4, pytest-9.0.3, pluggy-1.6.0 -- /home/aaron-mau/miniforge3/envs/gsoc/bin/python
cachedir: .pytest_cache
rootdir: /home/aaron-mau/Code/gsoc/mau/ioos_qc
configfile: pyproject.toml
plugins: anyio-4.13.0, cov-7.1.0
collecting ... collected 79 items

../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_location PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_location_bad_input PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_location_bbox PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_location_distance_threshold PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_single_location_nan PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_single_location_none PASSED
../ioos_qc/tests/test_qartod.py::QartodGrossRangeTest::test_gross_range_bad_input PASSED
../ioos_qc/tests/test_qartod.py::QartodGrossRangeTes

<ExitCode.OK: 0>

Nice! It looks like it ran, and now it lists 73 tests, 4 more than before. That makes sense, as we now see QartodImpossibleDateTest at the bottom of it. Checking our code coverage, we should be able to add the `cov` options from before.

(As I will be adding more tests in the future, this coverage will continue to change. I'll insert a snapshot of what this looks like below. Probably fine to minimize this or the previous cell in the future.)

```
============================= test session starts ==============================
platform linux -- Python 3.14.4, pytest-9.0.3, pluggy-1.6.0 -- /home/aaron-mau/miniforge3/envs/gsoc/bin/python
cachedir: .pytest_cache
rootdir: /home/aaron-mau/Code/gsoc/mau/ioos_qc
configfile: pyproject.toml
plugins: anyio-4.13.0, cov-7.1.0
collecting ... collected 73 items

../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_location PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_location_bad_input PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_location_bbox PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_location_distance_threshold PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_single_location_nan PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_single_location_none PASSED
../ioos_qc/tests/test_qartod.py::QartodGrossRangeTest::test_gross_range_bad_input PASSED
../ioos_qc/tests/test_qartod.py::QartodGrossRangeTest::test_gross_range_check PASSED
../ioos_qc/tests/test_qartod.py::QartodGrossRangeTest::test_gross_range_check_masked PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyPeriodTest::test_climatology_test_periods_day_of_year PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyPeriodTest::test_climatology_test_periods_monthly PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyPeriodTest::test_climatology_test_periods_quarter PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyPeriodTest::test_climatology_test_periods_week_of_year PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyPeriodFullCoverageTest::test_dayofyear_periods PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyPeriodFullCoverageTest::test_monthly_periods PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyPeriodFullCoverageTest::test_quarterly_periods PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyPeriodFullCoverageTest::test_weekofyear_periods PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_fspan_maximum PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_fspan_minimum PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_fspan_out_of_range_high PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_fspan_out_of_range_low PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_tspan_maximum PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_tspan_minimum PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_tspan_out_of_range_high PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_tspan_out_of_range_low PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_vspan_maximum PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_vspan_minimum PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_vspan_out_of_range_high PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_vspan_out_of_range_low PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_zspan_maximum PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_zspan_minimum PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_zspan_out_of_range_high PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_zspan_out_of_range_low PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyDepthTest::test_climatology_test_all_unknown PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyMissingTest::test_climatology_missing_values PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyTest::test_climatology_test PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyTest::test_climatology_test_depths PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyTest::test_climatology_test_fail PASSED
../ioos_qc/tests/test_qartod.py::QartodClimatologyTest::test_climatology_test_seconds_since_epoch PASSED
../ioos_qc/tests/test_qartod.py::QartodSpikeTest::test_spike PASSED
../ioos_qc/tests/test_qartod.py::QartodSpikeTest::test_spike_initial_final_values PASSED
../ioos_qc/tests/test_qartod.py::QartodSpikeTest::test_spike_masked PASSED
../ioos_qc/tests/test_qartod.py::QartodSpikeTest::test_spike_methods PASSED
../ioos_qc/tests/test_qartod.py::QartodSpikeTest::test_spike_negative_vals PASSED
../ioos_qc/tests/test_qartod.py::QartodSpikeTest::test_spike_realdata PASSED
../ioos_qc/tests/test_qartod.py::QartodSpikeTest::test_spike_test_bad_method PASSED
../ioos_qc/tests/test_qartod.py::QartodSpikeTest::test_spike_test_inputs PASSED
../ioos_qc/tests/test_qartod.py::QartodRateOfChangeTest::test_rate_of_change PASSED
../ioos_qc/tests/test_qartod.py::QartodRateOfChangeTest::test_rate_of_change_fail_flag PASSED
../ioos_qc/tests/test_qartod.py::QartodRateOfChangeTest::test_rate_of_change_missing_values PASSED
../ioos_qc/tests/test_qartod.py::QartodRateOfChangeTest::test_rate_of_change_negative_values PASSED
../ioos_qc/tests/test_qartod.py::QartodFlatLineTest::test_flat_line PASSED
../ioos_qc/tests/test_qartod.py::QartodFlatLineTest::test_flat_line_missing_values PASSED
../ioos_qc/tests/test_qartod.py::QartodFlatLineTest::test_flat_line_short_timeseries PASSED
../ioos_qc/tests/test_qartod.py::QartodFlatLineTest::test_flat_line_starting_from_beginning PASSED
../ioos_qc/tests/test_qartod.py::QartodFlatLineTest::test_flat_line_with_spike PASSED
../ioos_qc/tests/test_qartod.py::QartodAttenuatedSignalTest::test_attenuated_signal PASSED
../ioos_qc/tests/test_qartod.py::QartodAttenuatedSignalTest::test_attenuated_signal_missing PASSED
../ioos_qc/tests/test_qartod.py::QartodAttenuatedSignalTest::test_attenuated_signal_missing_time_window PASSED
../ioos_qc/tests/test_qartod.py::QartodAttenuatedSignalTest::test_attenuated_signal_range PASSED
../ioos_qc/tests/test_qartod.py::QartodAttenuatedSignalTest::test_attenuated_signal_time_window PASSED
../ioos_qc/tests/test_qartod.py::QartodDensityInversionTest::test_density_inversion_bad_depth_value PASSED
../ioos_qc/tests/test_qartod.py::QartodDensityInversionTest::test_density_inversion_down_up_cast_flags PASSED
../ioos_qc/tests/test_qartod.py::QartodDensityInversionTest::test_density_inversion_downcast_flags PASSED
../ioos_qc/tests/test_qartod.py::QartodDensityInversionTest::test_density_inversion_input PASSED
../ioos_qc/tests/test_qartod.py::QartodDensityInversionTest::test_density_inversion_one_record_input PASSED
../ioos_qc/tests/test_qartod.py::QartodDensityInversionTest::test_density_inversion_stable_depth_flags PASSED
../ioos_qc/tests/test_qartod.py::QartodDensityInversionTest::test_density_inversion_upcast_flags PASSED
../ioos_qc/tests/test_qartod.py::QartodUtilsTests::test_qartod_compare PASSED
../ioos_qc/tests/test_qartod.py::QartodImpossibleDateTest::test_all_nat PASSED
../ioos_qc/tests/test_qartod.py::QartodImpossibleDateTest::test_error_bad_datetime PASSED
../ioos_qc/tests/test_qartod.py::QartodImpossibleDateTest::test_future_year PASSED
../ioos_qc/tests/test_qartod.py::QartodImpossibleDateTest::test_span_good PASSED

============================== 73 passed in 0.32s ==============================
```

In [30]:
#   This output will change as I implement tests after this one.
pytest.main([
    "--assert=plain", "-v",
    "--cov-report=term-missing",
    "--cov=ioos_qc",
    testing_path,
])

============================= test session starts ==============================
platform linux -- Python 3.14.4, pytest-9.0.3, pluggy-1.6.0 -- /home/aaron-mau/miniforge3/envs/gsoc/bin/python
cachedir: .pytest_cache
rootdir: /home/aaron-mau/Code/gsoc/mau/ioos_qc
configfile: pyproject.toml
plugins: anyio-4.13.0, cov-7.1.0
collecting ... collected 79 items

../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_location PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_location_bad_input PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_location_bbox PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_location_distance_threshold PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_single_location_nan PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_single_location_none PASSED
../ioos_qc/tests/test_qartod.py::QartodGrossRangeTest::test_gross_range_bad_input PASSED
../ioos_qc/tests/test_qartod.py::QartodGrossRangeTes

/home/aaron-mau/miniforge3/envs/gsoc/lib/python3.14/site-packages/coverage/inorout.py:577: CoverageWarning: Module ioos_qc was previously imported, but not measured (module-not-measured); see https://coverage.readthedocs.io/en/7.14.1/messages.html#warning-module-not-measured
  self.warn(msg, slug="module-not-measured")


<ExitCode.OK: 0>

And it does look like we've improved testing coverage.
```shell
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/qartod.py                            389     72    81%   61-62, 83-84, 143-144, 153-154, 219-220, 238-239, 302, 311-328, 339-340, 346-347, 352-353, 358-359, 366-368, 456-459, 875-876, 896, 960-961, 968, 1072-1089, 1117-1140
```
Lines 1024-1042 are no longer showing up!

Now, to implement unit testing on the other new tests
* data_reception_test() - lines 1072-1089
    * We'll want to test both `from_time = None` and some timestamp.
    * Copy the "all_nat" test from impossible time
        * **Question for Filipe:** If this test is duplicated across classes, but still depends on "self", is there a better way to share between them?
            * Parametrized testing? I.e., `@pytest.mark.parametrize` before defining this function elsewhere
            * Does a base class make sense? Inherit "all_nat" from it in each of the following classes?
    * Do a successful dataset - where there are no flagged data
    * Do an unsuccessful one - where the data are flagged at some point
* time_gap_test() - lines 1117-1140
    * The "all_nat" test
    * A good one, i.e. where the gaps are less than the default 2 hours
        * Do another good one, where it isn't the default 2 hours
    * A bad one, where the last point is off by more than the tolerance
        * Should be asserting that the *last* point is flagged bad, not the second to last.
        * A bad one that handles the first point. Set the seconde point to be bad, and confirm that `flags[0] == QartodFlags.FAIL`

##### QartodDataReceptionTest

In [31]:
class QartodDataReceptionTest(unittest.TestCase):
    def setUp(self):
        times = [
            "2026-01-12T23:05:14.000000000",
            "2026-01-12T23:05:15.000000000",
            "2026-01-12T23:05:16.000000000",
            "2026-01-12T23:05:17.000000000",
        ]
        self.data_good = np.array(times, dtype="datetime64")    #   Shouldn't return any flags
        self.data_bad = self.data_good.copy()
        self.data_bad = np.append(
            self.data_bad,
            np.datetime64("2026-02-12T23:05:17.000000000")
        ) #   Add a point that is a day later
    
    def test_all_nat(self):
        dt = np.full(4, np.datetime64("NaT"))
        flags = qartod.impossible_date_test(tinp=dt)
        assert all(flags == 9)

    def test_none_bad(self):
        #   Should all be good if the `from_time` param is set within the default 6 hours away
        flags = qartod.data_reception_test(self.data_good, from_time="2026-01-12T23:10:00.000000000")
        assert all(flags == 1)

        #   `from_time = None` default test
        now = np.datetime64("now")  #   UTC
        # new_times = np.array([
        #     now - 4*(60*60),
        #     now - 3*(60*60),
        #     now - 2*(60*60),
        #     now - (60*60),  #   1 hour before current time
        # ])
        new_times = np.array([
            now - np.timedelta64(4, "h"),
            now - np.timedelta64(3, "h"),
            now - np.timedelta64(2, "h"),
            now - np.timedelta64(1, "h"),
        ])
        flags = qartod.data_reception_test(new_times)
        assert all(flags == 1)
        assert type(flags) == np.ma.core.MaskedArray

    def test_some_bad(self):
        flags = qartod.data_reception_test(self.data_bad, from_time="2026-02-12T23:10:00.000000000")
        assert flags[-1] == 1
        assert all(flags[0:3] == 4)  #   Should be bad, as they are more than 6 hours from the last point
        assert type(flags) == np.ma.core.MaskedArray

##### Potential redundancy
As per the question above to Filipe, I am testing again that the line `flag_arr[tinp.mask] = QartodFlags.MISSING` gets run in the `test_all_nat` function. This code is being reused in multiple testing classes, and I've been able to break this up before using Pytest [**parametrizations**](https://docs.pytest.org/en/stable/example/parametrize.html). The `pytest.parametrize` functions allow you to pull functions out or define data that can otherwise be used inside of multiple functions. However, I've never done this with Python Classes and I don't know if the "self" argument will create problems. The examples available at the link earlier in this cell aren't very helpful for helping me understand how to implement this in a Class setting.

This would resemble:

```python
@pytest.mark.parametrize("testname", [
    qartod.impossible_date_test,
    qartod.data_reception_test,
])
def test_all_nat(testname):
    dt = np.full(4, np.datetime64("NaT"))
    flags = testname(tinp=dt)
    assert np.all(flags == 9)

#   Then the rest of the classes, without `test_all_nat` in them.
```

Where `testname` represents each of the functions that we want to run on the data. In this case, both the `impossible_date_test` and `data_reception_test` (adding the `time_gap_test` later if we prefer).

Alternatively, we could use Python [Class Inheritance](https://www.geeksforgeeks.org/python/inheritance-in-python/). We could define a base "TimeBase" class, which includes `setUp` and `test_all_nat`, and then we define follow-up classes that would inherit these properties automatically. Like in the example link, we would get a new `QartodDataReceptionTest` class which would inherit `TimeBase` and, in theory, we wouldn't need to run anything. For all tests that inherit `TimeBase`, this test will run.

The child class would need to define the name of the test to run, since `test_all_nat` calls different functions to populate the flags, but that could be done in setUp.

```python
class TimeBase():
    def test_all_nat(self):
        dt = np.full(4, np.datetime64("NaT"))
        flags = self.testname(tinp=dt)
        assert np.all(flags == 9)

class QartodDataReceptionTest(TimeBase, unittest.TestCase):
    def setUp(self):
        times = [
            "2026-01-12T23:05:14.000000000",
            "2026-01-12T23:05:15.000000000",
            "2026-01-12T23:05:16.000000000",
            "2026-01-12T23:05:17.000000000",
        ]
        self.data_good = np.array(times, dtype="datetime64")
        self.data_bad = self.data_good.copy()
        self.data_bad = np.append(
            self.data_bad,
            np.datetime64("2026-02-12T23:05:17.000000000")
        )
        self.testname = qartod.data_reception_test
```

Let's get this incorperated once we've got all our test classes built. One to go: `time_gap_test`

##### QartodTimeGapTest

In [32]:
class QartodTimeGapTest(unittest.TestCase):
    def setUp(self):
        times = [
            "2026-01-12T23:05:14.000000000",
            "2026-01-12T23:05:15.000000000",
            "2026-01-12T23:05:16.000000000",
            "2026-01-12T23:05:17.000000000",
        ]
        self.data_good = np.array(times, dtype="datetime64")
        self.data_bad = self.data_good.copy()
        self.data_bad = np.append(
            self.data_bad,
            np.datetime64("2026-02-12T23:05:17.000000000")
        )

    def test_all_nat(self):
        dt = np.full(4, np.datetime64("NaT"))
        flags = qartod.time_gap_test(tinp=dt)
        assert all(flags == 9)

    def test_good(self):
        #   default 2 hours
        flags = qartod.time_gap_test(self.data_good)
        assert all(flags == 1)
        assert type(flags) == np.ma.core.MaskedArray

        #   non-default tolerance
        flags = qartod.time_gap_test(self.data_good, fail_span=0.5)
        assert all(flags == 1)

    def test_bad(self):
        #   Data gap on last point
        flags = qartod.time_gap_test(self.data_bad)
        assert flags[-1] == 4
        assert type(flags) == np.ma.core.MaskedArray

        #   Another run where bad_data[1] is flagged as a gap (shift all points), assert that flag[0] == flag[1]
        self.data_bad[1:-1] += np.timedelta64(3, "h")
        flags = qartod.time_gap_test(self.data_bad, fail_span=2.5)
        assert all(flags[2:4] == 1)
        assert flags[-1] == 4   #   last point should still be a gap
        assert flags[1] == 4    #   second point should be a gap, following the shift
        assert flags[0] == flags[1]

        #   And test if the second point is NaT - the first and third points should be flagged as UNKNOWN
        #   Added after the bug was found below.
        self.data_bad[1] = np.datetime64("NaT")
        flags = qartod.time_gap_test(self.data_bad, fail_span=2.5)
        assert all(flags.data == [2, 9, 2, 1, 4])


I noticed a bug when running through the code. If the second point is NaT, the first point should be returning a 9 - which is does. However, you can't really differentiate NaT points, and the third point (point 3 - NaT) is being flagged as 1. This is the initialized value, suggesting something isn't being run correctly.

Let's try this out by building our `data_bad` that we'll use in the unit tests.

In [33]:
times = [
    "2026-01-12T23:05:14.000000000",
    "2026-01-12T23:05:15.000000000",
    "2026-01-12T23:05:16.000000000",
    "2026-01-12T23:05:17.000000000",
]
data_good = np.array(times, dtype="datetime64")
data_bad = data_good.copy()
data_bad = np.append(
    data_bad,
    np.datetime64("2026-02-12T23:05:17.000000000")
)
print(data_bad) #   Shows us the bad data with a clear gap on the last point

['2026-01-12T23:05:14.000000000' '2026-01-12T23:05:15.000000000'
 '2026-01-12T23:05:16.000000000' '2026-01-12T23:05:17.000000000'
 '2026-02-12T23:05:17.000000000']


In [34]:
data_bad[1:-1] += np.timedelta64(3, "h")
print(data_bad) #   Apply the shift, as shown above. Now there is a gap on the first point.

['2026-01-12T23:05:14.000000000' '2026-01-13T02:05:15.000000000'
 '2026-01-13T02:05:16.000000000' '2026-01-13T02:05:17.000000000'
 '2026-02-12T23:05:17.000000000']


So if we visualize our flags at this stage, we should have two gaps: One at the end, one on the second point, and the first value should have taken on the value of the second point. So we should see 3 FAIL flags in the output.

In [35]:
print(qartod.time_gap_test(data_bad))

[4 4 1 1 4]


Now to finalize the test conditions where the second point is NaT.

In [36]:
data_bad[1] = np.datetime64("NaT")
print(data_bad)

['2026-01-12T23:05:14.000000000'                           'NaT'
 '2026-01-13T02:05:16.000000000' '2026-01-13T02:05:17.000000000'
 '2026-02-12T23:05:17.000000000']


When running this before, I'd get the following output
```python
flags = qartod.time_gap_test(data_bad)
flags
masked_array(data=[2, 9, 1, 1, 4],
             mask=False,
       fill_value=np.uint64(999999),
            dtype=uint8)
```
In this case, the first point is correctly updating to UNKNOWN, since it is unassessed. However, it says that the third point is GOOD, which isn't true. It should be UNKNOWN, since you can't take the diff of a NaT.

If we reconstruct our function and testing parameters, we'll see that there are 2 NaT values in our `diff_time`. So we should apply the flag UNKNOWN to the areas where the value itself exists (present in our `valid` array) but the value is NaT.

I think this looks like `flag_arr[valid & (diff_time == np.datetime64("NaT"))] = QartodFlags.UNKNOWN`, but we'll want to check.

In [37]:
tinp = np.ma.asarray(data_bad, dtype="datetime64[ns]").flatten()
tinp.mask = np.isnat(tinp.data)
flag_arr = np.ma.ones(tinp.size, dtype="uint8")
flag_arr[tinp.mask] = QartodFlags.MISSING
valid = ~tinp.mask
diff_time = np.ma.zeros(tinp.size, dtype="timedelta64[ns]")
diff_time[1:] = tinp.data[1:] - tinp.data[:-1]
diff_time

masked_array(data=[               0,            'NaT',            'NaT',
                         1000000000, 2667600000000000],
             mask=False,
       fill_value=NaT,
            dtype='timedelta64[ns]')

Actually, that's what the `mask` of a masked array is for.

In [38]:
diff_time.mask = np.isnat(diff_time.data)
diff_time.mask

array([False,  True,  True, False, False])

In [39]:
#   We want where both the data are valid (valid==True) and the diff_time `isnat` mask is True
print(valid)
print(diff_time.mask)
print(valid & diff_time.mask)   #   This should ONLY be true for the third point

[ True False  True  True  True]
[False  True  True False False]
[False False  True False False]


In [40]:
flag_arr[valid & diff_time.mask] = QartodFlags.UNKNOWN
print(flag_arr) #   The third point should be UNKNOWN, or flagged as 2 now

[1 9 2 1 1]


Now if I make the change in the code and reload the notebook, I should get a working result back of `[2 9 2 1 4]`

Update: I didn't. It turned out that my previous form of mapping my NaTs wasn't ideal.
* I found a new way to concatenate the data, row-wise using [`np.r_`](https://numpy.org/doc/stable/reference/generated/numpy.r_.html)
* Pair the valid points of `valid` (is the data not NaT) along with the new `prev_valid`
* Update the flag assignments to use what I found before with `diff_time.mask`

Now, I get `[2, 9, 2, 1, 4]` back, which is what we were expecting!

In [41]:
flags = qartod.time_gap_test(data_bad)
flags

masked_array(data=[2, 9, 2, 1, 4],
             mask=False,
       fill_value=np.uint64(999999),
            dtype=uint8)

#### Decorators
[**Decorators**](https://www.w3schools.com/PYTHON/python_decorators.asp) are very powerful in Python, as they add extra behaviors to the functions themselves.

Just about everything in python is an object. Even the functions themselves. As such, you can add properties to the functions you add.

Filipe pointed out an important decorator that is called before each of the QARTOD tests: `add_flag_metadata`. This lives in `utils`, but is used here to add metadata to the function's attributes.

In [42]:
#   Construct an array of 100 items, increasing by 1 each time. Then run the gross range test on it.
arr = qartod.gross_range_test(np.arange(100), fail_span=(2, 7), suspect_span=(4, 5))
arr

masked_array(data=[4, 4, 3, 3, 1, 1, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
                   4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
                   4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
                   4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
                   4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
                   4, 4, 4, 4, 4, 4, 4, 4, 4, 4],
             mask=False,
       fill_value=np.uint64(999999),
            dtype=uint8)

In [43]:
qartod.gross_range_test.standard_name

'gross_range_test_quality_flag'

In this instance, the `gross_range_test` now has an additional property called `standard_name`. It also has a `long_name`.

In [44]:
qartod.gross_range_test.long_name

'Gross Range Test Quality Flag'

For future review, we'll want to make sure that each function in the manual has code with an accompanying decorator for these additional properties. This particular decorator is pretty simple to add.
```python
@add_flag_metadata(
    standard_name="time_gap_test_flag",
    long_name="Time Gap Test Flag",
)
def time_gap_test(
    ...
)
```

In [45]:
#   For future reference
from ioos_qc.utils import add_flag_metadata

#### Another finding - duplicate timestamps
I was reading through the Rutgers glider QC and found a `check_duplicate_timestamps.py` file ([link here](https://github.com/gliderqctest/ruglider_qc/blob/master/check_duplicate_timestamps.py)), which looks for duplicate timestamps across multiple files. Though multiple files is probably out of the scope of QARTOD, we should be able to support flagging multiple identical timestamps in a file. There are examples of [xarray instances where this can occur](https://stackoverflow.com/questions/51058379/drop-duplicate-times-in-xarray), and I can see it causing problems later on when we differentiate the items in our arrays.

Not quite clear what we should do in this instance or where we should put this.

**Question for Filipe:** Where would you put a `duplicate_timestamp` test, and what would it do?
* Allow it to error out, or raise an exception.
* Print the warning to the log/console for the user to know that there is an issue.
* Flag one or all duplicate times as suspect (3) or fail (4).
* **A:** Check with the IOOS QC manual first and see what it says regarding duplicate timestamps.

From the manual:
> Note: For those gliders that do not update at regualar intervals, a large value for TIM_STMP can be assigned. **The gap check is not a solution for all timing errors.** Data could be measured or received earlier than expected. **This test does not address all clock drift/jump issues.**

This suggests that the manual isn't accounting for duplicate timestamps. Doing a ctrl+F of "duplicate" leads to no results in the manual. However, `utils.check_timestamps()` does contain what we're looking for.
* Checks that the times supplied are in monotonically increasing chronological order
    * Sorts the timestamps
    * Diffs the timestamps
    * Finds where the timestamps don't change
* Returns a boolean of test conditions from the sorted array, not including the points where the diff is 0, or where the sorted diff is more than the defined max time interval.

The `utils` version does not, however, return more than a single boolean. A single pass/fail. It isn't capable of returning flags as we envison it doing.

Considering this is not listed in the QARTOD manual either, Filipe suggests adding this to `argo.py`, which is a good place to include tests that are not present in the QARTOD manual.

In [46]:
#   I still have `data_bad` defined. Let's add a duplicate, such that our test will flag both the first
#   and the last value.
data_bad

array(['2026-01-12T23:05:14.000000000',                           'NaT',
       '2026-01-13T02:05:16.000000000', '2026-01-13T02:05:17.000000000',
       '2026-02-12T23:05:17.000000000'], dtype='datetime64[ns]')

In [47]:
data_bad = np.append(data_bad,np.datetime64("2026-01-12T23:05:14.000000000"))

[numpy.unique](https://numpy.org/doc/stable/reference/generated/numpy.unique.html) offers `return_inverse`, which it says is to "return the indices of the unique array that can be used to reconstruct [the array]".

In [48]:
uniques, inxs, inverse, counts = np.unique(data_bad, return_index=True, return_inverse=True, return_counts=True)

In [49]:
print(uniques)  #   Lists the unique values, like a lookup table. This is rearranged, with NaT at the end.
print(inxs)     #   The index where each unique value first appears. NaT is at the end, hence why 1 is at the end here
print(inverse)  #   Is used to reconstruct the original array by saying which unique value is where. len(inverse)=len(arr)
print(counts)   #   How many times each one arises. This tells us if the value in uniques is not present once.

['2026-01-12T23:05:14.000000000' '2026-01-13T02:05:16.000000000'
 '2026-01-13T02:05:17.000000000' '2026-02-12T23:05:17.000000000'
                           'NaT']
[0 2 3 4 1]
[0 4 1 2 3 0]
[2 1 1 1 1]


So we have `inverse`, which is correctly mapping each unique value to each position in the array.

We can combine that with `counts`, which will tell us how many times each of those values are present.

In [50]:
counts[inverse] 

array([2, 1, 1, 1, 1, 2])

In this case, we only care about where it is greater than 1. In theory, timestamps could repeat multiple times or be static, so there is no upper bound.

In [51]:
counts[inverse] > 1

array([ True, False, False, False, False,  True])

This is a boolean, which I can then pull out using [`np.where`](https://numpy.org/doc/stable/reference/generated/numpy.where.html).

In [52]:
np.where(counts[inverse] > 1)[0]
#   I don't know why it is giving me multiple dimensions, but I need the first one

array([0, 5])

So we just need to return uniques (it always is, not important), inverse, and counts.

In [53]:
@add_flag_metadata(
    standard_name="duplicate_timestamp_test_quality_flag",
    long_name="Duplicate Timestamp Test Quality Flag",
)
def duplicate_timestamp_test(
    tinp: Sequence[Real],
) -> np.ma.core.MaskedArray:
    """
    This test flags duplicate timestamps in the provided array.

    If duplicate timestamps are found, they are flagged as SUSPECT.

    Parameters
    ----------
    tinp
        Time input data as a numeric numpy array or list of real numbers.

    Returns
    -------
    flag_arr
        A masked array of flag values equal in size to that of the input `tinp`.
    """
    original_shape = tinp.shape
    tinp = np.ma.asarray(tinp, dtype="datetime64[ns]").flatten()
    flag_arr = np.ma.ones(tinp.size, dtype="uint8") #   Init to 1

    tinp.mask = np.isnat(tinp.data)
    flag_arr[tinp.mask] = QartodFlags.MISSING  #   Init missing timestamps to the missing flag
    valid = ~tinp.mask

    _, inverse, counts = np.unique(tinp, return_inverse=True, return_counts=True)
    duplicate_mask = counts[inverse] > 1
    flag_arr[valid & duplicate_mask] = QartodFlags.SUSPECT

    return flag_arr.reshape(original_shape)

In [54]:
#   Should return questionable values on index 0, 5 and a 9 on index 1.
duplicate_timestamp_test(data_bad)

masked_array(data=[3, 9, 1, 1, 1, 3],
             mask=False,
       fill_value=np.uint64(999999),
            dtype=uint8)

In `argo.py`, `numbers.Real` is imported as `N` so that's the only real change we need to make when we add this.

In adding the unit tests, I'm just going to include one test `test_all` that covers all the use cases at once, as done above.
* NaN in the second entry
* Duplicates in first and last

This should give 100% coverage of the new code.
```python
class ArgoDuplicateTimeTest(unittest.TestCase):
    def test_all(self):
        data = np.array([
            '2026-01-12T23:05:14.000000000',
            'NaT',
            '2026-01-13T02:05:16.000000000',
            '2026-01-13T02:05:17.000000000',
            '2026-02-12T23:05:17.000000000',
            '2026-01-12T23:05:14.000000000'],
            dtype="datetime64[ns]"
        )
        flags = argo.duplicate_timestamp_test(data)
        assert flags[0] == flags[-1] == 3
        assert flags[1] == 9
        assert all(flags[2:4] == 1)
        assert type(flags == np.ma.core.MaskedArray)
```

All the items are now present in the code and are ready to go. Remember to run `ruff` on each of the modified files to match the formatting of the repo.

We will leave the PR open until next week to get community feedback before merging.

##  Syntax test
* How is it described in the manual?
* How is it represented in the code?
* Does anything need to be done in either?

### Manual description
The manual describes the syntax test as a **required** test that will "Check to ensure that the message is structured properly". The closest match for this test elsewhere is described in Argo test #1 (Platform Identification Test), which originates from the TESAC Argo format that preceeded the BUFR format (page 10 of Wong et al. 2025). If this test failed, then Argo wouldn't transmit that data message.

So this test originates from data streams. It checks for full data messages, and proposes two checks:
[]: # TIL how to do an alphanumeric list in markdown, and make a comment
<ol type="a">
  <li>Confirm that the fixed-length message has the correct number of characters received `REC_CHAR`</li>
  <li>Passes a standard parity bit check, cyclic redundancy check, or other check.</li>
</ol>

The syntax test description leaves a lot up to the user.

> "Many such syntax tests exist, and the user should select the best criteria for one or more syntax tests."

Furthermore, it states that there are lots of different ways that the operators handle these flawed messages, as a flawed message doesn't necessarily mean it will fail in their processing.
* This suggests that this test shouldn't "error out".

Next, the manual states that the check is performed **only at the message level**, indicating that flags are usually reported at some higher level than a single variable. While this could be applied as a **point flag** (see below), all tests should be writing out a more generic flag array as it is.

The manual explicitly states that nothing within the message itself is tested, just the message itself, and this can potentially include multiple messages in a *data record*.
* In this instance, *data record* could be a matrix of data, or a series of individual lines. So this test should work on both a *line* of data, and a *series of lines*.

In the example, only two flags are output. Either the line passes (QARTOD GOOD=1), or it fails (QARTOD FAIL=4). It only fails if $REC\_CHAR \neq NCHAR$. For example, the flag for line $n$ of a data record $dr$ fails if $REC\_CHAR_{dr(n)} = 126$, but $NCHAR = 128$.

#### Community
In my previous ping to the UG2 community, the syntax test is a little confusing and has discouraged users from implementing it.

When discussing with QARTOD users in a meeting setting, they highlighted that there are many different sources of data with loads of potential data structures to be expected. In many cases, each gets its own processing code to handle the syntax as well as the contents therein. They did emphasize that, if implemented, the user's specification to the data stream will be paramount to its adoption for this reason.

On the plus side, I think I've potentially found a copy of the "raw" data that would otherwise be getting parsed. This is a tricky test, because we need example data that potentially preceed manufacturer QC. Having recently met with members of the [Spray Glider lab in San Diego](https://spraydata.ucsd.edu/), whom build their own gliders, I learned that the glider data is assembled from JSON files into a NetCDF. These data are published online, though the level 1 products can be hard to find. I did managed to find glider data from the California Underwater Glider Network [here](https://data.caloos.org/#platform/01336c63-359d-532d-bfc9-3adee4491503/v2?tab=visualization). At this page, the compressed JSON can be downloaded by hitting "downloads" and then selecting `data.json.gz`. The dataset grows from 3.24 MB to 21.9 MB when extracted, but it is still very much usable for testing purposes.
* Update: This isn't raw data `.JSON` (`import json`). It already contains QARTOD flags. It may be the `.dbf` file that is available at the ShapeFile link?
* Update: Not in the `.dbf` either, there are QARTOD flags in it (`pip install dbf`).

#### Point flags
For QARTOD, this may make sense a **point flag** level, such that the point flag represents the quality of the measurement at a unique index or timestamp for all sensors is influenced. In my mind, point flags can be built from an accumulation of too many bad flags in important data columns or be manually assigned by a user.

Examples of point flags
* A voltage spike from the battery or controller. Something that can travel through multiple sensors.
* When one of the coordinates are bad. If the position of the measurement in time and space is unknown, then it carries very little value to science on finer scales, regardless of what the measurement is.
* Something is known to be wrong with the measurement. For example, a glider has become entangled in fishing line and is not diving correctly, giving errating measurements that go against the manufacturer's indended use. Biofouling is another potential example. Time spent in the air also applies.

In each of these cases, the point flag would be suspect or bad, telling the user "though there are data at these points, and the streams themselves could be good, the measurement itself is flawed and we advise against you using it". Some data centers remove data based on a point flag value.

### Code
After going through the timing/gap tests previously, I've learned a little more about some of the `ioos_qc` libraries and modules available. Off the top of my head, I know there could be something in `utils`, and potentially `argo` if not already in `qartod`.

* `qartod.py` doesn't contain anything that looks like what the manual suggests.
* `argo.py` contains additional tests, but nothing like the syntax test.
* `utils.py` contains `isfixedlength()`, which checks if a list is the correct length. This is similar to `utils.check_timestamps()` before, which only returns a single boolean.
    * It doesn't *flag anything*, it just returns the bool True if it doesn't error out. It actually does what the manual suggests it shouldn't do, which is error out (`TypeError` if not a sequence, `ValueError` if wrong length).

I looked through the Argo QC manual and the repository `dm_floats` is mentioned, suggesting that there could be preexisting MATLAB code that would just need to be translated. However, this repository is either not public or present on the [Argo DMQC](https://github.com/orgs/ArgoDMQC/repositories), and the other repos don't quite fit the description of it.

`dm_floats` *does* exist, however, on the Euro-Argo ERIC (euroargodev) [group page](https://github.com/euroargodev/dm_floats/tree/master). This fits the description of the manual, but it may take some digging to find an actual function. **It looks like they have some fun [online schools!](https://euroargodev.github.io/argoonlineschool/Lessons/L02_TheArgoData/Chapter23_QualityControl.html)**
* The only "real" mention of a "Platform identification test" is made in `print_history.m`, which suggests that the quality control is run elsewhere.
* There is a mention of Coriolis-Argo in the code, so I Googled "Coriolis-Argo float real time qc" and found the SEANOE page for the [processing chain software](https://www.seanoe.org/data/00345/45589/). Considering so much of glider data is modeled from the Argo program, it makes sense to check in here for documentation and examples of data formatting.
    * folder 1: Float configurations.
    * folder 2: Documentation.
        * The `argo_coriolis_matlab_decoder...pdf` section 6.1.1 describes that Argo data uses the HEX (hexadecimal) format.
        * Section 7 uses the DAC decoder for real-time data.
        * However, there is no mention of a "Platform identification test". Or "platform" at all in this context.
        * There is an [APEX PROFILER USER MANUAL](https://www.aoml.noaa.gov/phod/argo/webpage_sections/doc/float_types/documentation/APEX_TS45/Job-34266.1%20APEX%20User%20Manual%20102015.pdf) which specifies a little information about how the HEX file is formatted, as well as the telemetry error check (cyclic redundancy check as described in the QARTOD test description).
    * Folder 3: Software, all written in MATLAB.
        * Look for `TEST001_PLATFORM_IDENTIFICATION`

```matlab
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
% TEST 1: platform identification test
%
if (testFlagList(1) == 1)
   % always Ok
   testDoneList(1, :) = 1;
end
```

It doesn't look like it actually does anything. `testFlagList` gets built from a loop over the `expectedTestList`, which lists `TEST001_PLATFORM_IDENTIFICATION` as the first test. There are multiple `add_rtqc...` configuration files where this is defined, but in each case it's just marked as "done". Other tests, like TEST 20 (questionable position test) has actual code.

Bummer. Not our problem, we should be able to move on with something generic.

### Actions
* Ping external contacts about "raw" data that they use this test on
* Implement the `syntax` test
    * Take in a sequence, potentially a sequence of sequences, and if so, loop through each dimension to populate a 1xn array of flags like before.
        * Note: We don't know if this is done on "character" characters or "bytes". A 12-character HEX represents 6 bytes.
    * If possible, follow the APEX user manual to do the CRC test. This is on the CRC byte, which is usually a float and the first byte of the message. See page 25 of 36 in the [2017 manual](https://www.aoml.noaa.gov/phod/argo/webpage_sections/doc/float_types/documentation/APEX_TS45/Job-34266.1%20APEX%20User%20Manual%20102015.pdf).

#### Note about CRC tests
Having read the APEX user manual helps me understand what some of the data look like in AUVs prior to communication via satellite. [**Cyclic redundancy checks**](https://en.wikipedia.org/wiki/Cyclic_redundancy_check) seem to be helpful for error detection, usually done by calculating the remaining bytes of a message. The remainder of a division on the amount of data is recorded, such that the length of the data is preserved. This length is saved in a byte, which is usually at the beginning of the line for floats (but [at the end](https://youtu.be/_x0vbnUKYSU?si=kbWchKUjFt6NS5Re&t=373) for most other applications it seems). When the data is transmitted, the original data and the CRC byte are sent together. If the receiver works on the data and doesn't get the value saved in the CRC byte, then there was some amount of data that was lost.

Analogy from my computer science friend:
> Imagine you write "12" on top of a box of 12 eggs. Then you give them to a friend. If your friend opens the box and counts 11 eggs, then something is off. Something happened to the packet of eggs.

So these are very common machine-to-machine tests to verify that data arrives intact. It's like a [checksum](https://en.wikipedia.org/wiki/Checksum), [but more sensitive](https://www.geeksforgeeks.org/computer-networks/difference-between-checksum-and-crc/). Good when your data is really valuable and you have a robust system in place.

In [58]:
test = ["CF010502AF02470085010101169217199E9401AD85091F48979B004662240E"]   #   Appendix A in the Argo manual
np.array(test[0])   #   Type to try and convert to is "<U62"
len(test[0])

62

In [71]:
@add_flag_metadata(
    standard_name = "syntax_test_quality_flag",
    long_name = "Syntax Test Quality Flag",
)
def syntax_test(
    inp: Sequence[str],
    nchar: Real,
    lentype: str = "char",
) -> np.ma.core.MaskedArray:
    """Checks for raw data formatting and flags if data are the incorrect length.

    For an array of raw data (either a single line of HEX or rows of messages), it loops through the number of lines
    and assigns a flag GOOD (1) if nchar equals the specified length of length type `lentype` or flag FAIL (4) if
    not equal.

    Parameters
    ----------
    inp
        Raw data sequence to be counted.
    nchar
        Designated tolerance or number of characters expected for testing.
    lentype
        A string designating either "char" or "byte" to clarify unit for counting. Defaults to "char".

    Returns
    -------
    flag_arr
        A masked array of flag values equal in size to that of the input.

    Example
    -------
    flags = syntax_test()
    """
    #   Start by finding out the dimensions of the input - it could be a single line of HEX chars or rows of lines of HEX chars
    original_shape = inp.shape
    inp = np.ma.asarray(inp, dtype="<U62").flatten() #   Type
    flag_arr = np.ma.ones(inp.size, dtype="uint8")
 
    lentype = lentype.strip().lower()
    if lentype not in ("char", "byte"):
        raise ValueError(f"lentype must be 'char' or 'byte', got: {lentype}")

    for i in range(inp.shape[0]):
        rec_char = len(inp[i])
        if lentype == "byte":
            #   HEX is 2 characters - don't run if odd number
            if rec_char % 2 != 0:
                flag_arr[i] = QartodFlags.FAIL
                continue
            rec_char = rec_char / 2
        #   else it's "char" and we keep the same length
        if nchar == rec_char:
            flag_arr[i] = QartodFlags.GOOD
        else:
            flag_arr[i] = QartodFlags.FAIL
    
    return flag_arr.reshape(original_shape)

In [75]:
print(syntax_test(np.array(test[0]), nchar=62, lentype="char"))
print(syntax_test(np.array(test[0]), nchar=61, lentype="char"))
print(syntax_test(np.array(test[0]), nchar=31, lentype="byte"))
print(syntax_test(np.array(test[0]), nchar=30, lentype="byte"))

1
4
1
4


I think this does what we want it to do. Need to find examples of more packets of a longer length to prove it really works.

In [77]:
print(syntax_test(np.full((4,1), test[0]), nchar=62, lentype="char"))

[[1]
 [1]
 [1]
 [1]]


In [78]:
type(syntax_test(np.full((4,1), test[0]), nchar=62, lentype="char"))

numpy.ma.MaskedArray